# TensorFlow を使ってみる

**学習目標**
  1. 定数テンソルの定義と基本演算の実行を練習する
  1. TensorFlow の自動微分機能を使用する
  1. TensorFlow を使用してゼロから線形回帰をトレーニングする方法を学ぶ

このノートブックでは、まず TensorFlow におけるテンソルの主な操作を確認し、TensorFlow の変数（Variable）を操作する方法を理解します。これらが Python 組み込みのリストや NumPy 配列とどのように互換性があるかを説明します。

次に、勾配降下法を用いてゼロから線形回帰をトレーニングする問題に進みます。最初に取り組むべきことは、関数（ここでは損失）の引数（ここではモデルの重み）に対する勾配を計算する方法を理解することです。これを可能にする TensorFlow の構成要素が `tf.GradientTape` であり、これについて説明します。

最後に、線形モデルから生成された合成データを使用して、1 次元の線形回帰の重みを学習するシンプルなトレーニング ループを作成します。

ボーナス演習として、非線形モデルから生成されたデータに対しても同様のことを行い、線形モデルのパフォーマンスを向上させるために非線形特徴量を手動でエンジニアリングします。

In [ ]:
import os
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

import numpy as np
import tensorflow as tf
from matplotlib import pyplot as plt

In [ ]:
print(tf.__version__)

## テンソルに対する演算

### 変数と定数

TensorFlow のテンソルは、定数（`tf.constant`）または変数（`tf.Variable`）のいずれかです。

定数の値は変更できませんが、変数の値は変更できます。

主な違いは、`tf.Variable` のインスタンスには値を変更できるメソッドがあるのに対し、`tf.constant` で構築されたテンソルにはこれらのメソッドがなく、したがってその値を変更できないことです。`tf.Variable` の値を変更したい場合は、以下のいずれかのメソッドを使用します。
* `x.assign(new_value)`
* `x.assign_add(value_to_be_added)`
* `x.assign_sub(value_to_be_subtracted)`

In [ ]:
x = tf.constant([2, 3, 4])
x

In [ ]:
x = tf.Variable(2.0, dtype=tf.float32, name="my_variable")

**演習:** `assign(..)` メソッドを使用して、上で作成した変数 `x` に新しい値を代入します。各操作の後に `x` を出力して、値がどのように変化するかを確認してください。

In [ ]:
x.assign(None)  # TODO: ここにコードを記述してください。
x

In [ ]:
x.assign_add(None)  # TODO: ここにコードを記述してください。
x

In [ ]:
x.assign_sub(None)  # TODO: ここにコードを記述してください。
x

### 要素ごとの演算

TensorFlow は、NumPy と同様の要素ごとの（point-wise）テンソル演算を提供しています。
* `tf.add`: テンソルの要素同士を加算
* `tf.multiply`: テンソルの要素同士を乗算
* `tf.subtract`: テンソルの要素同士を減算
* `tf.math.*`: テンソルの要素に適用される通常の数学演算
* その他多数...

標準的な算術演算のほとんど（`tf.add`、`tf.subtract` など）は、対応する通常の算術記号（`+`、`-` など）でオーバーロードされています。

**演習:** 2 つの TensorFlow 定数 `a = [5, 3, 8]` と `b = [3, -1, 2]` を作成します。次に、以下を計算してください。
1. `tf.add` と `+` を使用して定数 `a` と `b` の和を計算し、両方の演算が同じ値を生成することを確認します。
2. `tf.multiply` と `*` を使用して定数 `a` と `b` の積を計算し、両方の演算が同じ値を生成することを確認します。
3. `tf.math.exp` を使用して定数 `a` の指数関数を計算します。なお、この演算には型を指定する必要があることに注意してください。

In [ ]:
a = None  # TODO: ここにコードを記述してください。
b = None  # TODO: ここにコードを記述してください。
c = None  # TODO: ここにコードを記述してください。
d = None  # TODO: ここにコードを記述してください。

print("c:", c)
print("d:", d)

In [ ]:
a = None  # TODO: ここにコードを記述してください。
b = None  # TODO: ここにコードを記述してください。
c = None  # TODO: ここにコードを記述してください。
d = None  # TODO: ここにコードを記述してください。

print("c:", c)
print("d:", d)

In [ ]:
# tf.math.exp は浮動小数点数を想定しているため、明示的に型を指定する必要があります
a = None  # TODO: ここにコードを記述してください。
b = None  # TODO: ここにコードを記述してください。

print("b:", b)

### NumPy との相互運用性

ネイティブの TF テンソルに加えて、TensorFlow の演算はネイティブの Python 型や NumPy 配列をオペランドとして受け取ることができます。

In [ ]:
# ネイティブの Python リスト
a_py = [1, 2]
b_py = [3, 4]

**演習:** `tf.add` を使用して、ネイティブの Python 配列 `a` と `b` の和を計算します。

In [ ]:
# TODO: ここにコードを記述してください。

In [ ]:
# NumPy 配列
a_np = np.array([1, 2])
b_np = np.array([3, 4])

**演習:** `tf.add` を使用して、NumPy 配列 `a` と `b` の和を計算します。

In [ ]:
# TODO: ここにコードを記述してください。

In [ ]:
# ネイティブの TF テンソル
a_tf = tf.constant([1, 2])
b_tf = tf.constant([3, 4])

**演習:** `tf.add` を使用して、TensorFlow 定数 `a` と `b` の和を計算します。

In [ ]:
# TODO: ここにコードを記述してください。

`.numpy()` を使用して、ネイティブの TF テンソルを NumPy 配列に変換できます。

In [ ]:
a_tf.numpy()

## 線形回帰

それでは、低レベルの TensorFlow 演算を使用して線形回帰を実装してみましょう。

コースの後半では、高レベルの TensorFlow を使用してこれをより抽象化された方法で行う方法を学びます。

### トイ データセット

次の関数をモデル化します。

\begin{equation}
y= 2x + 10
\end{equation}

In [ ]:
X = tf.constant(range(10), dtype=tf.float32)
Y = 2 * X + 10

print(f"X:{X}")
print(f"Y:{Y}")

モデルを評価するためのテスト データセットも作成しましょう。

In [ ]:
X_test = tf.constant(range(10, 20), dtype=tf.float32)
Y_test = 2 * X_test + 10

print(f"X_test:{X_test}")
print(f"Y_test:{Y_test}")

#### 損失関数

構築できる最もシンプルなモデルは、x の各値に対してトレーニング セットの標本平均を返すモデルです。

In [ ]:
y_mean = Y.numpy().mean()


def predict_mean(X):
    y_hat = [y_mean] * len(X)
    return y_hat


Y_hat = predict_mean(X_test)

平均二乗誤差（MSE）を使用すると、損失は次のようになります。
\begin{equation}
MSE = \frac{1}{m}\sum_{i=1}^{m}(\hat{Y}_i-Y_i)^2
\end{equation}

このシンプルなモデルの場合、損失は次のようになります。

In [ ]:
errors = (Y_hat - Y_test) ** 2
loss = tf.reduce_mean(errors)
loss.numpy()

上記の MSE 損失の値は、より複雑なモデルのパフォーマンスを比較するためのベースラインになります。

ここで、線形回帰モデルを使用したときのモデルの予測を含むベクトルを $\hat{Y}$ と表すと、
\begin{equation}
\hat{Y} = w_0X + w_1
\end{equation}

モデルの係数を引数として取る損失関数を記述できます。

In [ ]:
def loss_mse(X, Y, w0, w1):
    Y_hat = w0 * X + w1
    errors = (Y_hat - Y) ** 2
    return tf.reduce_mean(errors)

### 勾配関数

勾配降下法を使用するには、各重みに対する損失関数の偏微分を求める必要があります。手動で微分を計算することもできますが、TensorFlow の自動微分機能を使えばその必要はありません。

勾配降下法を行っている間、損失はパラメータ $w_0$ と $w_1$ の関数であると考えます。したがって、これらの変数に対する偏微分を計算したいと考えます。

そのために、勾配情報を記録する `tf.GradientTape` インスタンスのコンテキスト内で損失計算をラップする必要があります。

```python
with tf.GradientTape() as tape:
    loss = # 計算
```

これにより、後で `tf.GradientTape` コンテキスト内で計算された任意のテンソルの、`tf.Variable` インスタンスに対する勾配を計算できるようになります。

```python
gradients = tape.gradient(loss, [w0, w1])
```

モデルの重みに対する損失の勾配を計算することで、この手順を説明します。

**ラボタスク #6:** 以下の関数を完成させ、モデルの重み `w0` と `w1` に対する損失の勾配を計算してください。

In [ ]:
def compute_gradients(X, Y, w0, w1):
    # TODO: ここにコードを記述してください。

In [ ]:
w0 = tf.Variable(0.0)
w1 = tf.Variable(0.0)

dw0, dw1 = compute_gradients(X, Y, w0, w1)

In [ ]:
print("dw0:", dw0.numpy())

In [ ]:
print("dw1", dw1.numpy())

### トレーニング ループ

ここでは、収束する非常にシンプルなトレーニング ループを作成します。簡単にするために、バッチ処理、個別のテスト セットの作成、重みのランダム初期化などのベスト プラクティスは無視していることに注意してください。

**ラボタスク #7:** 以下の `for` ループを完成させて線形回帰をトレーニングします。
1. `compute_gradients` を使用して `dw0` と `dw1` を計算します。
2. 次に、計算された勾配値と `LEARNING_RATE` を使用して、`.assign_sub(...)` メソッドで `w0` と `w1` の値を再代入します。
3. 最後に、100 ステップごとに `loss` を計算して出力します。上で作成した `loss_mse` 関数を使用して `loss` を計算してください。

In [ ]:
STEPS = 1000
LEARNING_RATE = 0.02
MSG = "STEP {step} - loss: {loss}, w0: {w0}, w1: {w1}\n"


w0 = tf.Variable(0.0)
w1 = tf.Variable(0.0)


for step in range(1, STEPS + 1):
    dw0, dw1 = None  # TODO: ここにコードを記述してください。
    # TODO: ここにコードを記述してください。
    # TODO: ここにコードを記述してください。

    if step % 100 == 0:
        loss = None  # TODO: ここにコードを記述してください。
        print(MSG.format(step=step, loss=loss, w0=w0.numpy(), w1=w1.numpy()))

では、この線形回帰のテスト損失と、常にトレーニング セットの平均値を出力するベースライン モデルのテスト損失を比較してみましょう。

In [ ]:
loss = loss_mse(X_test, Y_test, w0, w1)
loss.numpy()

確かに大幅に改善されています。

## ボーナス

$y=xe^{-x^2}$ のような非線形関数のモデル化に挑戦してみましょう。

In [ ]:
X = tf.constant(np.linspace(0, 2, 1000), dtype=tf.float32)
Y = X * tf.exp(-(X**2))

In [ ]:
%matplotlib inline

plt.plot(X, Y)

In [ ]:
def make_features(X):
    f1 = tf.ones_like(X)  # バイアス。
    f2 = X
    f3 = tf.square(X)
    f4 = tf.sqrt(X)
    f5 = tf.exp(X)
    return tf.stack([f1, f2, f3, f4, f5], axis=1)

In [ ]:
def predict(X, W):
    return tf.squeeze(X @ W, -1)

In [ ]:
def loss_mse(X, Y, W):
    Y_hat = predict(X, W)
    errors = (Y_hat - Y) ** 2
    return tf.reduce_mean(errors)

In [ ]:
def compute_gradients(X, Y, W):
    with tf.GradientTape() as tape:
        loss = loss_mse(X, Y, W)
    return tape.gradient(loss, W)

In [ ]:
STEPS = 2000
LEARNING_RATE = 0.02


Xf = make_features(X)
n_weights = Xf.shape[1]

W = tf.Variable(np.zeros((n_weights, 1)), dtype=tf.float32)

# プロット用
steps, losses = [], []
plt.figure()


for step in range(1, STEPS + 1):
    dW = compute_gradients(Xf, Y, W)
    W.assign_sub(dW * LEARNING_RATE)

    if step % 100 == 0:
        loss = loss_mse(Xf, Y, W)
        steps.append(step)
        losses.append(loss)
        plt.clf()
        plt.plot(steps, losses)


print(f"STEP: {STEPS} MSE: {loss_mse(Xf, Y, W)}")

plt.figure()
plt.plot(X, Y, label="actual")
plt.plot(X, predict(Xf, W), label="predicted")
plt.legend()

Copyright 2021 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.